python -m notebook

In [1]:
from tests.conftest import *

usecase = baseline_usecase.__wrapped__()
usecase.streams

[Stream(name='stream-0', type='isochronous', interval=100000, maxFramesPerInterval=1, maxFrameSize=126, maxLatency=500000, tlkrPssblPlcmnt=['host-2'], lstnsrPssblPlcmnt=[['host-0']], earliest_transmit_offset=0, latest_transmit_offset=50000, jitter=0, offset=None, accLatency=None, pathLists=[['host-2.Port1', 'switch-c0.Port2', 'host-0']], lifespanCycles=None),
 Stream(name='stream-1', type='isochronous', interval=100000, maxFramesPerInterval=2, maxFrameSize=167, maxLatency=10000000, tlkrPssblPlcmnt=['host-1'], lstnsrPssblPlcmnt=[['host-6']], earliest_transmit_offset=0, latest_transmit_offset=50000, jitter=0, offset=None, accLatency=None, pathLists=[['host-1.Port1', 'switch-c0.Port1', 'switch-c1.Port2', 'switch-c2.Port2', 'switch-c3.Port3', 'host-6']], lifespanCycles=None),
 Stream(name='stream-2', type='isochronous', interval=100000, maxFramesPerInterval=1, maxFrameSize=96, maxLatency=1000000, tlkrPssblPlcmnt=['host-1'], lstnsrPssblPlcmnt=[['host-6']], earliest_transmit_offset=0, latest

In [2]:
usecase.specification.get_config()

'line-10-5-1000-10-1-1000-6|5-100-1-1-1-True-True-True-50-5-False-0.74-0.25-False-10-False-5-20-False-100-200-300-1-3-64-256-0.5-1.0-2.0-5.0-10.0'

In [26]:
import pytest
pytest.main(["-q", "tests/test_stream.py::test_stream_paths_are_valid", "-s"])

.
============================== warnings summary ===============================
C:\Users\hamzchah\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\_pytest\cacheprovider.py:475
  C:\Users\hamzchah\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\_pytest\cacheprovider.py:475: PytestCacheWarning: could not create cache path \\nas-a1\redirected$\hamzchah\Desktop\TSN_optimizer_platform\.pytest_cache\v\cache\nodeids: [WinError 1251] This operation is supported only when you are connected to the server: '\\\\nas-a1\\redirected$\\hamzchah\\Desktop\\TSN_optimizer_platform\\pytest-cache-files-ifu4wt1c'
    config.cache.set("cache/nodeids", sorted(self.cached_nodeids))

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
1 passed, 1 warning in 0.17s


<ExitCode.OK: 0>

In [ ]:
def build_pruned_graph(self, graph: nx.DiGraph):
    """
    Simple pruning rule:

    IF pathLists exists:
        keep only ports in pathLists (and their edges)
        → everything else is implicitly removed

    ELSE:
        remove concurrent ports of:
            - tlkrPssblPlcmnt (talkers)
            - lstnsrPssblPlcmnt (listeners)
    """

    from topology_handler import TopologyHandler

    # --------------------------------------------------
    # CASE 1: path-based pruning (STRICT MODE)
    # --------------------------------------------------
    if self.pathLists:

        active_ports = set()

        # keep only ports in pathLists (exclude last element = listener node)
        for path in self.pathLists:
            for hop in path[:-1]:
                active_ports.add(hop)

        # remove everything not in active_ports
        pruned = nx.DiGraph()
        pruned.add_nodes_from(graph.nodes(data=True))
        for u, v, data in graph.edges(data=True):
            src_port = data.get("sourcePort")
            if src_port in active_ports:
                pruned.add_edge(u, v, **data)

        isolated_nodes = [n for n, d in pruned.degree() if d < 1]
        if isolated_nodes:
            pruned.remove_nodes_from(isolated_nodes)

        return pruned

    # --------------------------------------------------
    # CASE 2: free routing mode (concurrency pruning)
    # --------------------------------------------------
    edges_to_remove = []

    # talker concurrency removal
    edges_to_remove += TopologyHandler.get_concurrent_ports(
        graph,
        self.tlkrPssblPlcmnt,
        typ="talker"
    )

    # listener concurrency removal
    edges_to_remove += TopologyHandler.get_concurrent_ports(
        graph,
        [l for sub in self.lstnsrPssblPlcmnt for l in sub],
        typ="listener"
    )

    # rebuild graph
    pruned = nx.DiGraph()
    pruned.add_nodes_from(graph.nodes(data=True))

    for u, v, data in graph.edges(data=True):
        if (u, v, data) in edges_to_remove:
            continue
        pruned.add_edge(u, v, **data)

    isolated_nodes = [n for n, d in pruned.degree() if d < 1]
    if isolated_nodes:
        pruned.remove_nodes_from(isolated_nodes)

    return pruned